# Buzzlytics — YOLOv8 Fine-Tuning (Google Colab, dataset from Google Drive)

Fine-tunes **YOLOv8n** on the unified 4-class bee dataset: `bee`, `pollen_bee`, `varroa_bee`, `wasp`.

## One-time prep (on your machine)
1. Zip the merged dataset folder so the zip contains `train/ valid/ test/`, each with `images/` and `labels/`:
   - Folder to zip: `Buzzlytics---CSCI435/datasets/data`
   - Windows: right-click `data` → Send to → Compressed (zip). Name it `bee_dataset.zip`.
2. Upload `bee_dataset.zip` to **Google Drive**.
3. Right-click it in Drive → **Share** → **Anyone with the link** → **Copy link**.

## Run
4. Runtime → Change runtime type → **GPU** (free T4 is enough).
5. Paste the Drive share link into `DATASET_URL` in the first code cell.
6. Run all cells.

The notebook downloads the zip from Drive, extracts it, auto-locates the dataset, trains, reports per-class mAP, and downloads `best.pt`.

**Class balance note:** the set is imbalanced — varroa and bee common, **pollen sparse (~110 instances)**, wasp ~1000. Expect lower mAP on `pollen_bee`; document as a known limitation in the report.

In [ ]:
# === 0. Configuration ===
# Paste your Google Drive SHARE LINK to bee_dataset.zip (Anyone-with-the-link).
# e.g. https://drive.google.com/file/d/1AbCdEfGhIjKlMnOpQ/view?usp=sharing
DATASET_URL = "PASTE_GDRIVE_SHARE_LINK_HERE"

EPOCHS = 100
IMGSZ = 640
BATCH = 16
MODEL = "yolov8n.pt"   # nano: justified by the 10-FPS requirement

assert DATASET_URL != "PASTE_GDRIVE_SHARE_LINK_HERE", "Paste your Google Drive share link first"

In [ ]:
# === 1. Confirm GPU ===
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

In [ ]:
# === 2. Install dependencies ===
!pip -q install "ultralytics==8.4.71" gdown pyyaml

In [ ]:
# === 3. Download the dataset zip from Google Drive + extract ===
import gdown, zipfile
from pathlib import Path
from collections import Counter

gdown.download(url=DATASET_URL, output="/content/bee_dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("/content/bee_dataset.zip") as z:
    z.extractall("/content/dataset")

# Auto-locate the dataset root (the dir that directly contains train/images),
# regardless of how the zip nested its folders.
DATA_DIR = None
for p in Path("/content/dataset").rglob("train/images"):
    DATA_DIR = str(p.parent.parent)
    break
assert DATA_DIR, "Could not find a train/images folder inside the zip. Re-zip so it contains train/ valid/ test/ each with images/ + labels/."
print("DATA_DIR:", DATA_DIR)

# Sanity check: class distribution (0=bee 1=pollen 2=varroa 3=wasp)
c = Counter()
for txt in Path(DATA_DIR).rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            c[int(line.split()[0])] += 1
print("instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path(DATA_DIR) / s / "images"
    print(f"{s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")

# --- Fallback if gdown fails (very large files / Drive quota) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/bee_dataset.zip" /content/bee_dataset.zip
# then re-run the extract + locate lines above.

In [ ]:
# === 4. Write the dataset yaml (absolute path to the extracted data) ===
import yaml
data_cfg = {
    "path": DATA_DIR,            # set in the previous cell after extraction
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": {0: "bee", 1: "pollen_bee", 2: "varroa_bee", 3: "wasp"},
}
with open("/content/bee_dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open("/content/bee_dataset.yaml").read())

In [ ]:
# === 5. Fine-tune YOLOv8n ===
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data="/content/bee_dataset.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="bee_detector",
    patience=25,          # early stop if no val improvement
    seed=42,
    # augmentations (report these in the write-up):
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, degrees=5.0,
)

In [ ]:
# === 6. Validate + per-class metrics (for the report's results table) ===
best = "runs/detect/bee_detector/weights/best.pt"
metrics = YOLO(best).val(data="/content/bee_dataset.yaml", split="test")
print("\nmAP@50:     ", round(float(metrics.box.map50), 4))
print("mAP@50-95:  ", round(float(metrics.box.map), 4))
names = {0: "bee", 1: "pollen_bee", 2: "varroa_bee", 3: "wasp"}
print("\nPer-class mAP@50:")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"  {names.get(int(i), i):<12} {float(ap):.4f}")

In [ ]:
# === 7. Download best.pt ===
# Place the downloaded file at  cv_pipeline/weights/best.pt  in your local repo.
from google.colab import files
files.download("runs/detect/bee_detector/weights/best.pt")

# Also save the training plots (confusion matrix, PR curves) for the report:
!ls runs/detect/bee_detector/*.png
# files.download('runs/detect/bee_detector/confusion_matrix.png')  # uncomment to grab